# Veriden Tahmine: Diyabet Hastalığı İlerleme Skoru

## Proje amacı
Bu çalışmada, başlangıç ölçümlerindeki üç sayısal değişkenden yararlanarak diyabet hastalarında bir yıl sonraki hastalık ilerleme skorunu tahmin eden bir **regresyon** modeli kurulmuştur. Çalışma eğitim amaçlıdır; klinik karar veya tanı aracı değildir.

## Veri kaynağı
Scikit-learn içinde hazır gelen `load_diabetes` veri kümesi kullanılmıştır. Veri kümesi 442 gözlem ve 10 özellik içerir. Bu projede yorumlanabilirliği artırmak için üç özellik seçilmiştir: BMI, kan basıncı (bp) ve s5. Hedef değişken, bir yıl sonraki hastalık ilerleme skorudur. Özellikler kaynak veri kümesinde standartlaştırılmıştır.

Kaynak: Scikit-learn `load_diabetes` veri kümesi dokümantasyonu — https://scikit-learn.org/stable/modules/generated/sklearn.datasets.load_diabetes.html


In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.datasets import load_diabetes
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score


## 1. Veriyi yükleme ve proje veri setini oluşturma

In [2]:
veri = load_diabetes(as_frame=True)
df = veri.frame.copy()

df = df.rename(columns={
    "target": "ilerleme_skoru",
    "bmi": "bmi_standart",
    "bp": "kan_basinc_standart",
    "s5": "kan_biyobelirteci_s5"
})

ozellikler = ["bmi_standart", "kan_basinc_standart", "kan_biyobelirteci_s5"]
hedef = "ilerleme_skoru"
proje_df = df[ozellikler + [hedef]].copy()

print("Veri boyutu:", proje_df.shape)
display(proje_df.head())


Veri boyutu: (442, 4)


,bmi_standart,kan_basinc_standart,kan_biyobelirteci_s5,ilerleme_skoru
0,0.061696,0.021872,0.019907,151.0
1,-0.051474,-0.026328,-0.068332,75.0
2,0.044451,-0.005670,0.002861,141.0
3,-0.011595,-0.036656,0.022688,206.0
4,-0.036385,0.021872,-0.031988,135.0


## 2. Veri sözlüğü ve veri kalitesi

In [3]:
veri_sozlugu = pd.DataFrame({
    "Değişken": ["bmi_standart", "kan_basinc_standart", "kan_biyobelirteci_s5", "ilerleme_skoru"],
    "Tür": ["Sayısal", "Sayısal", "Sayısal", "Sayısal / hedef"],
    "Açıklama": [
        "Standartlaştırılmış vücut kitle indeksi göstergesi",
        "Standartlaştırılmış ortalama kan basıncı göstergesi",
        "Standartlaştırılmış serum ölçümünün s5 değişkeni",
        "Başlangıçtan bir yıl sonraki hastalık ilerleme skoru"
    ]
})
display(veri_sozlugu)
print("Eksik değer sayıları:\n", proje_df.isnull().sum())
print("Tam yinelenen satır:", proje_df.duplicated().sum())


,Değişken,Tür,Açıklama
0,bmi_standart,Sayısal,Standartlaştırılmış vücut kitle indeksi göster...
1,kan_basinc_standart,Sayısal,Standartlaştırılmış ortalama kan basıncı göste...
2,kan_biyobelirteci_s5,Sayısal,Standartlaştırılmış serum ölçümünün s5 değişkeni
3,ilerleme_skoru,Sayısal / hedef,Başlangıçtan bir yıl sonraki hastalık ilerleme...


Eksik değer sayıları:
 bmi_standart            0
kan_basinc_standart     0
kan_biyobelirteci_s5    0
ilerleme_skoru          0
dtype: int64
Tam yinelenen satır: 0


### Temizleme açıklaması
- Veri kümesi Scikit-learn tarafından hazır ve sayısal biçimde sağlanır.
- Seçilen dört sütunda eksik değer ve yinelenen satır kontrolü yapılır.
- Kaynak veri kümesinde özellikler standartlaştırılmıştır; bu nedenle yeniden ölçekleme uygulanmamıştır.
- Model hattında eksik değer oluşması ihtimaline karşı medyan ile doldurma adımı tutulmuştur.


## 3. Keşifsel veri analizi (EDA)

In [4]:
print("İlk satırlar:")
display(proje_df.head())
print("Boyut:", proje_df.shape)
print("Özet istatistikler:")
display(proje_df.describe().T)


İlk satırlar:


,bmi_standart,kan_basinc_standart,kan_biyobelirteci_s5,ilerleme_skoru
0,0.061696,0.021872,0.019907,151.0
1,-0.051474,-0.026328,-0.068332,75.0
2,0.044451,-0.005670,0.002861,141.0
3,-0.011595,-0.036656,0.022688,206.0
4,-0.036385,0.021872,-0.031988,135.0


Boyut: (442, 4)
Özet istatistikler:


,count,mean,std,min,25%,50%,75%,max
bmi_standart,442.0,-2.245564e-16,0.047619,-0.090275,-0.034229,-0.007284,0.031248,0.170555
kan_basinc_standart,442.0,-4.797570e-17,0.047619,-0.112399,-0.036656,-0.005670,0.035644,0.132044
kan_biyobelirteci_s5,442.0,9.268604e-17,0.047619,-0.126097,-0.033246,-0.001947,0.032432,0.133597
ilerleme_skoru,442.0,1.521335e+02,77.093005,25.000000,87.000000,140.500000,211.500000,346.000000


In [5]:
plt.figure(figsize=(7, 4.5))
plt.scatter(proje_df["bmi_standart"], proje_df[hedef], alpha=0.55)
plt.title("BMI göstergesi ile hastalık ilerleme skoru")
plt.xlabel("BMI (standartlaştırılmış değer)")
plt.ylabel("İlerleme skoru")
plt.show()

bmi_gruplari = pd.qcut(proje_df["bmi_standart"], q=4, labels=["Q1", "Q2", "Q3", "Q4"])
grup_ozeti = proje_df.assign(BMI_grubu=bmi_gruplari).groupby("BMI_grubu", observed=False)[hedef].mean()
display(grup_ozeti)

grup_ozeti.plot(kind="bar", figsize=(7, 4.5), title="BMI çeyrek gruplarına göre ortalama ilerleme skoru")
plt.xlabel("BMI grubu (Q1 en düşük – Q4 en yüksek)")
plt.ylabel("Ortalama ilerleme skoru")
plt.xticks(rotation=0)
plt.show()


/tmp/ipykernel_1739/73396217.py:6: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


BMI_grubu
Q1    103.415929
Q2    123.890909
Q3    172.287037
Q4    210.108108
Name: ilerleme_skoru, dtype: float64

/tmp/ipykernel_1739/73396217.py:16: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


### EDA gözlemleri
- Veri seti 442 satırdan ve bu proje için seçilmiş 4 sütundan oluşur.
- BMI göstergesi ile hedef skor arasında genel olarak pozitif yönlü bir ilişki görülür; ancak noktaların yayılımı ilişkinin tek başına güçlü bir tahmin kuralı olmadığını gösterir.
- BMI çeyrek gruplarının ortalama hedef skorları karşılaştırılarak gruplar arasındaki fark incelenir.
- Üç özellikli model, veri kümesindeki tüm bilgiyi kullanmadığı için tahmin hatası beklenir.


## 4. Özellik ve hedef ayrımı, eğitim/test bölünmesi

In [6]:
X = proje_df[ozellikler]
y = proje_df[hedef]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42
)
print("Eğitim X:", X_train.shape, "| Eğitim y:", y_train.shape)
print("Test X:", X_test.shape, "| Test y:", y_test.shape)


Eğitim X: (353, 3) | Eğitim y: (353,)
Test X: (89, 3) | Test y: (89,)


## 5. Model seçimi ve eğitimi

In [7]:
model = Pipeline([
    ("eksik_deger", SimpleImputer(strategy="median")),
    ("regresyon", LinearRegression())
])
model.fit(X_train, y_train)
print("Model eğitimi tamamlandı.")


Model eğitimi tamamlandı.


### Model seçiminin gerekçesi
Hedef değişken sayısal bir ilerleme skorudur; bu nedenle problem regresyondur. Linear Regression, başlangıç düzeyinde anlaşılması ve katsayılarının yorumlanabilmesi nedeniyle seçilmiştir. Model üç sayısal özellik kullanır. Daha karmaşık algoritmalarla karşılaştırma bu projenin kapsamı dışındadır.


## 6. Tahmin ve değerlendirme

In [8]:
tahmin = model.predict(X_test)
mae = mean_absolute_error(y_test, tahmin)
rmse = mean_squared_error(y_test, tahmin) ** 0.5
r2 = r2_score(y_test, tahmin)

print(f"MAE : {mae:.2f}")
print(f"RMSE: {rmse:.2f}")
print(f"R²  : {r2:.3f}")


MAE : 44.29
RMSE: 53.77
R²  : 0.454


### Metriklerin yorumu
- **MAE:** Tahminlerin gerçek skordan mutlak ortalama sapmasını gösterir; hedef değişkenin kendi ölçeğindedir.
- **RMSE:** Büyük hataları daha fazla cezalandırır.
- **R²:** Test verisindeki hedef değişken varyansının model tarafından açıklanan oranını özetler. Negatif R², ortalama tahmininden daha kötü performansa işaret edebilir.


## 7. Üç yeni örnek için tahmin

In [9]:
yeni_ornekler = pd.DataFrame([
    [-0.05, -0.02, -0.03],
    [ 0.02,  0.01,  0.04],
    [ 0.08,  0.05,  0.07]
], columns=ozellikler)

yeni_sonuclar = model.predict(yeni_ornekler)
sonuc_tablosu = yeni_ornekler.copy()
sonuc_tablosu["tahmin_edilen_ilerleme_skoru"] = yeni_sonuclar
display(sonuc_tablosu)


,bmi_standart,kan_basinc_standart,kan_biyobelirteci_s5,tahmin_edilen_ilerleme_skoru
0,-0.05,-0.02,-0.03,98.705134
1,0.02,0.01,0.04,187.205088
2,0.08,0.05,0.07,252.358409


## 8. Akıllı tahmin aracı (mini uygulama)

In [10]:
def tahmin_et(bmi_standart, kan_basinc_standart, kan_biyobelirteci_s5):
    """Standartlaştırılmış üç girdi ile ilerleme skorunu tahmin eder."""
    giris = pd.DataFrame([{
        "bmi_standart": bmi_standart,
        "kan_basinc_standart": kan_basinc_standart,
        "kan_biyobelirteci_s5": kan_biyobelirteci_s5
    }])
    skor = model.predict(giris)[0]
    return f"Tahmin edilen ilerleme skoru: {skor:.1f}"

# Örnek kullanım
print(tahmin_et(-0.05, -0.02, -0.03))


Tahmin edilen ilerleme skoru: 98.7


## 9. Sonuç, sınırlılıklar ve geliştirme önerileri
Bu çalışma, üç standartlaştırılmış başlangıç ölçümünü kullanarak bir yıl sonraki hastalık ilerleme skorunu tahmin eden temel bir doğrusal regresyon modeli kurmuştur. Modelin test başarısı MAE, RMSE ve R² ile raporlanmıştır.

**Sınırlılıklar**
- Yalnızca üç özellik kullanılmıştır; veri kümesindeki diğer değişkenler dışarıda bırakılmıştır.
- Doğrusal model, doğrusal olmayan ilişkileri yeterince yakalayamayabilir.
- Veri kümesi görece küçüktür ve belirli bir araştırma örneklemine dayanır.
- Tahminler klinik tanı, tedavi veya bireysel sağlık kararı için kullanılamaz.

**Geliştirme önerileri**
- Tüm uygun özelliklerle modelin yeniden kurulması,
- Ridge/Lasso ve Decision Tree gibi modellerle karşılaştırma,
- Çapraz doğrulama ve hiperparametre değerlendirmesi,
- Hata dağılımı ve gerçek-tahmin grafiği eklenmesi.


## 10. Etkileşimli kullanım (terminal)

Notebook hücresinde örnek tahminler gösterilir. Serbest kullanıcı girdisi için aynı klasörde `python model_ve_tahmin.py` komutunu çalıştırın; program BMI, bp ve s5 değerlerini sorar.

In [11]:
print("Örnek kullanıcı girdisi sonucu:", tahmin_et(0.00, 0.01, 0.02))

Örnek kullanıcı girdisi sonucu: Tahmin edilen ilerleme skoru: 164.4


## 10. Etkileşimli kullanım (terminal)

Notebook hücresinde örnek tahminler gösterilir. Serbest kullanıcı girdisi için aynı klasörde `python model_ve_tahmin.py` komutunu çalıştırın; program BMI, bp ve s5 değerlerini sorar.

In [12]:
print("Örnek kullanıcı girdisi sonucu:", tahmin_et(0.00, 0.01, 0.02))

Örnek kullanıcı girdisi sonucu: Tahmin edilen ilerleme skoru: 164.4
